### LLM Monitoring with LangSmith & Langfuse

**flan-t5-large + LangChain** stack

Two options:
- **Option A — LangSmith**: native LangChain, zero code changes
- **Option B — Langfuse**: open-source, works with any Python app

### 0. Install dependencies

In [3]:
#Run once — comment out after first install
#!pip install langsmith langchain langchain-core langchain-community transformers
#!pip install langfuse  # Option B only

#### 1. Shared setup — model + LangChain pipeline

In [3]:
import warnings
warnings.filterwarnings('ignore')

from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate

model_name = 'google/flan-t5-large'
#Online Mode
#tokenizer  = AutoTokenizer.from_pretrained(model_name)
#model      = AutoModelForSeq2SeqLM.from_pretrained(model_name)

#Offline Mode
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True
)

hf_pipeline = pipeline(
    'text2text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    do_sample=False
)

# Wrap in LangChain so chains, callbacks and tracers work
llm = HuggingFacePipeline(pipeline=hf_pipeline)
print('Model loaded:', model_name)

Device set to use cpu


Model loaded: google/flan-t5-large


C:\Users\Ajay\AppData\Local\Temp\ipykernel_16260\2743708921.py:32: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


2. Application to monitor
A two-step pipeline: style transfer then summarise.

In [4]:
#Create a reusable prompt template.
style_template = ChatPromptTemplate.from_template(
    'Rewrite the following text in a {style} style:\n\n{text}'
)
summary_template = ChatPromptTemplate.from_template(
    'Summarise the following in one sentence:\n\n{text}'
)

#orchestrating multiple LLM calls.
def run_pipeline(user_text: str, style: str = 'formal') -> dict:
    # Step 1 — style transfer
    style_prompt = style_template.format_messages(style=style, text=user_text)
    styled       = llm.invoke(style_prompt[0].content)
    # Step 2 — summarise
    summary_prompt = summary_template.format_messages(text=styled)
    summary        = llm.invoke(summary_prompt[0].content)
    return {'styled': styled, 'summary': summary}

# Quick smoke test (no monitoring yet)
result = run_pipeline('I am super excited about the new AI tools coming out!!')
print('Styled: ', result['styled'])
print('Summary:', result['summary'])

Styled:  I am super excited about the new AI tools coming out!!
Summary: I am excited about the new AI tools coming out!



Option A — LangSmith
LangSmith is built into LangChain. Setting three env vars enables automatic tracing — **no code changes needed**.

**Setup:** Sign up at https://smith.langchain.com → create a project → copy API key.

In [5]:
import os

os.environ['LANGCHAIN_TRACING_V2'] = 'true'          # enables auto-tracing
#When LangChain sees this it automatically intercepts every LangChain operation.
#Code > LangChain > Langsmith recorder > LLM
# from smith.langchain.com to upload traces.
LANGCHAIN_API_KEY    = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_PROJECT']    = 'flan-t5-demo'  # project name in the UI

print('LangSmith tracing active')

LangSmith tracing active


A1. Auto-tracing — zero code changes
-   Just run the same pipeline. Every `llm.invoke()` call is captured automatically.

In [6]:
# Identical call to Section 2 — LangSmith intercepts it transparently
result = run_pipeline(
    user_text='The new gaming console launches next month with 4K support.',
    style='casual'
)
print('Styled: ', result['styled'])
print('Summary:', result['summary'])

# Go to smith.langchain.com to see:
# - Each LLM call as a span with latency
# - Input / output for every step
# - Total run time
'''Every LLM call is recorded and we would have
The UI shows

prompt
completion
token usage
latency
errors '''

Styled:  The new gaming console launches next month with 4K support.
Summary: Sony has unveiled a new gaming console called the PlayStation 4.


'Every LLM call is recorded and we would have\nThe UI shows\n\nprompt\ncompletion\ntoken usage\nlatency\nerrors '

A2. Named runs — add metadata for filtering
-   langsmith.trace() lets you tag runs with names, user IDs and custom metadata.
-   run_type="chain" which tells LangSmith "This span represents an entire workflow."
-   Common run types
    -   chain (Entire Pipeline)
    -   llm (LLM Call)
    -   tool (External tool)
    -   retriever (Vector search)
    -   embedding (embedding Model)
    -   parser (output parser)
-   Later in LangSmith you can filter
    -   Show only formal requests.
    -   Show only casual requests.
    -   Show only production.
    -   Show only beta users.
-   Metadata is searchable


In [7]:
from langsmith import trace
import time

def run_pipeline_traced(user_text: str, style: str = 'formal', user_id: str = 'anon') -> dict:
    #Manual tracing insted of relying on automatic tracing
    with trace(
        name     = 'style-transfer-pipeline',
        run_type = 'chain',
        tags     = ['style-transfer', style],
        metadata = {'user_id': user_id, 'style': style}
    ):
        t0 = time.time()

        with trace(name='step-1-style', run_type='llm'):
            style_prompt = style_template.format_messages(style=style, text=user_text)
            styled = llm.invoke(style_prompt[0].content)

        with trace(name='step-2-summarise', run_type='llm'):
            summary_prompt = summary_template.format_messages(text=styled)
            summary = llm.invoke(summary_prompt[0].content)

        print(f'Total latency: {(time.time()-t0)*1000:.0f} ms')

    return {'styled': styled, 'summary': summary}

result = run_pipeline_traced(
    user_text='Deep learning models are getting cheaper to run every year.',
    style='formal',
    user_id='user_42'
)
print(result)


Total latency: 8522 ms
{'styled': 'Deep learning models are getting cheaper to run every year.', 'summary': 'Deep learning models are getting cheaper to run every year.'}


metadata={
    "user_id":user_id,
    "style":style
}

Metadata is searchable.
Production examples:
-   tenant
-   customer
-   language
-   region
-   subscription
-   model_version
-   prompt_version
-   experiment
-   browser
-   device
-   release

Example:
metadata={
    "model":"gpt-4o",
    "version":"v5",
    "country":"Germany",
    "premium":True
}

-   Nested traces
Pipeline
    │
    ├── Step 1
    │
    └── Step 2
-   which can show which step is slow instead of Pipeline took 2.5 sec
-   Prompt formatting : 20ms
-   Retriever: 90ms
-   LLM: 1800ms
-   Parser: 10ms

Client() : creates a LangSmith Client
list_runs(): returns previous traces
Finally we attach evaluation to a run


##### A3. Collect user feedback
Attach a thumbs-up/down score to any run — feeds the Feedback tab and fine-tuning datasets.

In [8]:
from langsmith import Client

ls_client = Client()  # picks up LANGCHAIN_API_KEY from env

# List the 5 most recent runs
runs = list(ls_client.list_runs(
    project_name=os.environ['LANGCHAIN_PROJECT'],
    limit=5
))

print(f'Found {len(runs)} recent runs')
for r in runs:
    duration = (r.end_time - r.start_time) if r.end_time else 'running'
    print(f'  {r.name:35s}  latency={duration}  id={r.id}')

# Attach a score to the most recent run (simulating a user thumbs-up)
if runs:
    ls_client.create_feedback(
        run_id  = runs[0].id,
        key     = 'user_rating',
        score   = 1,            # 1 = positive, 0 = negative
        comment = 'Output was clear and concise'
    )
    print('\nFeedback submitted for run:', runs[0].id)

Found 5 recent runs
  HuggingFacePipeline                  latency=0:00:05.087485  id=019fbf9b-f809-7a61-b3c9-51ba9fc447d5
  step-2-summarise                     latency=0:00:05.087485  id=df93d66b-773b-4471-92f2-01dae940dd3f
  HuggingFacePipeline                  latency=0:00:05.143910  id=019fbf9b-e3e8-7763-b48c-bae4ba8a9e61
  step-1-style                         latency=0:00:05.143910  id=38f44db5-4820-4d23-8793-7a626092bfbc
  style-transfer-pipeline              latency=0:00:10.231395  id=63db13b2-7d73-4ee0-8370-74f8573cbd1d

Feedback submitted for run: 019fbf9b-f809-7a61-b3c9-51ba9fc447d5


##### A4. Aggregate metrics — latency summary
Pull stats from the API for a quick dashboard view.

In [9]:
import statistics
from datetime import datetime, timedelta, timezone

recent_runs = list(ls_client.list_runs(
    project_name=os.environ['LANGCHAIN_PROJECT'],
    start_time=datetime.now(timezone.utc) - timedelta(hours=1),
    limit=20
))

latencies = []
for r in recent_runs:
    if r.end_time and r.start_time:
        latencies.append((r.end_time - r.start_time).total_seconds() * 1000)

print(f'Runs in last 1h : {len(recent_runs)}')
if latencies:
    print(f'Avg latency (ms): {statistics.mean(latencies):.0f}')
    print(f'p95 latency (ms): {sorted(latencies)[int(len(latencies)*0.95)]:.0f}')
    print(f'Min / Max (ms)  : {min(latencies):.0f} / {max(latencies):.0f}')
else:
    print('No completed runs yet — run the cells above first')

Runs in last 1h : 0
No completed runs yet — run the cells above first



##### Option B — Langfuse

Langfuse is **open-source** and model-agnostic — works with any Python app. Can be self-hosted (Docker) or used via cloud.

**Setup:** Sign up at https://cloud.langfuse.com → create project → copy Public Key, Secret Key.

In [ ]:
import os

os.environ['LANGFUSE_PUBLIC_KEY'] = 'pk-lf-YOUR_PUBLIC_KEY'
os.environ['LANGFUSE_SECRET_KEY'] = 'sk-lf-YOUR_SECRET_KEY'
os.environ['LANGFUSE_HOST']       = 'https://cloud.langfuse.com'  # or self-hosted URL

from langfuse import Langfuse
langfuse = Langfuse()
print('Langfuse connected:', langfuse.auth_check())

##### B1. Manual tracing — full control over spans
Langfuse uses a **trace → span** model. A trace = one user interaction; spans = individual steps inside it.

In [ ]:
import time

def run_pipeline_langfuse(user_text: str, style: str = 'formal', session_id: str = 's1') -> dict:
    trace = langfuse.trace(
        name       = 'style-transfer-pipeline',
        session_id = session_id,
        metadata   = {'style': style},
        tags       = [style]
    )

    # Step 1 — style transfer
    span1 = trace.span(name='step-1-style', input={'text': user_text, 'style': style})
    t0    = time.time()
    style_prompt = style_template.format_messages(style=style, text=user_text)
    styled       = llm.invoke(style_prompt[0].content)
    span1.end(output={'result': styled}, metadata={'latency_ms': (time.time()-t0)*1000})

    # Step 2 — summarise
    span2 = trace.span(name='step-2-summarise', input={'text': styled})
    t1    = time.time()
    summary_prompt = summary_template.format_messages(text=styled)
    summary        = llm.invoke(summary_prompt[0].content)
    span2.end(output={'result': summary}, metadata={'latency_ms': (time.time()-t1)*1000})

    trace.update(output={'summary': summary})
    langfuse.flush()  # ensure data is sent before notebook exits

    return {'styled': styled, 'summary': summary}

result = run_pipeline_langfuse(
    user_text  = 'The economy grew by 3% last quarter despite high inflation.',
    style      = 'casual',
    session_id = 'notebook-demo'
)
print('Styled: ', result['styled'])
print('Summary:', result['summary'])

##### B2. LangChain callback — auto-tracing (like LangSmith)
Langfuse's `CallbackHandler` gives you automatic tracing with no manual span management.

In [ ]:
from langfuse.callback import CallbackHandler

handler = CallbackHandler(
    session_id='langchain-auto-trace',
    tags=['auto-traced']
)

# Pass as a callback — Langfuse captures every LLM call automatically
prompt   = style_template.format_messages(style='pirate', text='Please hold for customer support.')
response = llm.invoke(prompt[0].content, config={'callbacks': [handler]})

langfuse.flush()
print('Response:', response)

##### B3. Score a generation (user feedback)

In [ ]:
# Fetch the most recent trace
traces = langfuse.get_traces(limit=1)

if traces.data:
    trace_id = traces.data[0].id
    langfuse.score(
        trace_id = trace_id,
        name     = 'user_rating',
        value    = 0.9,          # 0.0 to 1.0
        comment  = 'Output was fluent and accurate'
    )
    langfuse.flush()
    print(f'Scored trace {trace_id} with 0.9')
else:
    print('No traces found — run pipeline cells above first')

##### Option A vs Option B — summary

| | LangSmith | Langfuse |

| **Integration** | Native LangChain — env vars only | Manual spans or LangChain callback |
| **Auto-tracing** | Yes — zero code changes | Via `CallbackHandler` |
| **Hosting** | SaaS only | SaaS or self-hosted (Docker) |
| **Open source** | No | Yes (MIT) |
| **Feedback / scores** | `create_feedback()` | `langfuse.score()` |
| **Best for** | Pure LangChain apps | Any Python app; self-hosted needs |

**Recommendation :** use LangSmith — least friction since already on LangChain. Switch to Langfuse if you want open-source or plan to self-host.